# Importador Histórico Unificado MER v2 — preparación

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Importador_Historicos_Unificado_MER_v2.ipynb)

**Versión inicial: no escribe en Google Sheets.** Ejecuta el programa versionado en `scripts/importador_historicos_v2.py`, construye las seis tablas en SQLite y CSV y produce `informe.json`. Antes de habilitar la carga a la nueva base MER v2 se deben validar los campos del ZIP de licitaciones y la identidad de proveedores. No cargar los archivos generados automáticamente a producción.

**Archivos de ejemplo:** `2026-9.zip` (licitaciones), `COT_2026-09.zip` (Compra Ágil), `2026-9 (OC).zip` (OC).

In [ ]:
!git clone -q https://github.com/juliopez/MercadoPublico-Inteligencia.git /content/MercadoPublico-Inteligencia
from google.colab import files
print('Sube los tres ZIP oficiales del mismo mes, uno tras otro')
subidos=files.upload()
print('Recibidos:',list(subidos))

In [ ]:
import pathlib,subprocess,sys
nombres={'licitaciones':'2026-9.zip','cot':'COT_2026-09.zip','oc':'2026-9 (OC).zip'}
# Ajusta nombres SOLO si los ZIP oficiales descargados usan otra denominación.
for tipo,nombre in nombres.items():
 assert pathlib.Path(nombre).is_file(),f'Falta {tipo}: {nombre}'
cmd=[sys.executable,'/content/MercadoPublico-Inteligencia/scripts/importador_historicos_v2.py',
 '--licitaciones',nombres['licitaciones'],'--cot',nombres['cot'],'--oc',nombres['oc'],
 '--out','/content/preparado_mercado_publico_mer_v2']
subprocess.run(cmd,check=True)
print('Preparación terminada: no se han modificado Google Sheets')

In [ ]:
from google.colab import files
import json,zipfile,pathlib
carpeta=pathlib.Path('/content/preparado_mercado_publico_mer_v2')
print(json.dumps(json.loads((carpeta/'informe.json').read_text()),indent=2,ensure_ascii=False))
salida='/content/preparado_mercado_publico_mer_v2.zip'
with zipfile.ZipFile(salida,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in carpeta.iterdir():
  if f.suffix in ('.csv','.json'):z.write(f,f.name)
files.download(salida)
print('Se descargaron seis CSV y un informe. SQLite se conserva temporalmente en Colab.')